# 03 - Baseline: Item2Vec treinado e avaliado (Fase 2 · ML · Task 4)
**Squad 2 — Negócio: ML em Streaming | P2 · Recomendação de produtos no carrinho (Item2Vec)**
**Responsável:** Zaiden Emiliano Segundo Seleme · **Branch:** `feat/squad2-fase2-ml-zaiden`

---

### Objetivo
Primeira versão **funcional** do modelo: treinar o Item2Vec nas cestas de treino, recomendar o top-5 para cada carrinho do hold-out e medir o resultado. Não é o modelo final; é a referência sobre a qual as melhorias serão medidas.

### Regras desta versão
* **Parâmetros fixados antes de olhar o teste** (os do one page). Ajustar parâmetros olhando o hold-out inflaria as métricas; a busca de parâmetros fica para a etapa de melhorias, com uma validação separada dentro do treino.
* **Mesma prova para todos os modelos:** `gold/item2vec_holdout` (336 cestas, 1 item removido por cesta).
* **Comparação obrigatória** com o baseline ingênuo "top-5 mais vendidos". O Item2Vec só se justifica se superá-lo.

### Entradas e saídas
| Lê (Gold, notebook 02) | Grava (Gold) |
|---|---|
| `item2vec_cestas_treino` (frases), `item2vec_holdout` (teste), `item2vec_skus` (popularidade e subcategoria) | `item2vec_vetores` (o modelo: 1 vetor por SKU), `item2vec_recomendacoes_holdout` (top-5 de cada carrinho), `item2vec_avaliacao` (métricas por versão, em append) |

O modelo é salvo como **tabela Delta de vetores**, e não como arquivo do gensim. Assim a inferência em streaming (notebook 04) lê os vetores com Spark e calcula o cosseno sem depender do gensim.

In [ ]:
%pip install gensim==4.4.0 python-dotenv==1.0.1 --quiet

In [ ]:
dbutils.library.restartPython()

## 0. Ambiente, caminhos e parâmetros do modelo

In [ ]:
import os
from datetime import datetime
from dotenv import load_dotenv, find_dotenv
import numpy as np
import pandas as pd
import gensim
from gensim.models import Word2Vec
from pyspark.sql import functions as F
from pyspark.sql.functions import col

# Padronização de TimeZone Oficial da Squad 2 (Horário de Brasília - UTC-3)
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

dotenv_path = find_dotenv()
if not dotenv_path:
    candidatos = [
        os.path.join(os.getcwd(), ".env"),
        os.path.join(os.path.dirname(os.getcwd()), ".env"),
        os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), ".env"),
        "/Workspace/Shared/estagio_ed/.env",
        "/Workspace/Repos/estagio_ed/.env",
    ]
    dotenv_path = next((c for c in candidatos if os.path.isfile(c)), None)
load_dotenv(dotenv_path, override=True)

storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME")
client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")
assert storage_account and client_id and tenant_id and client_secret, "Credenciais do Azure ADLS não encontradas no .env"

fqdn = f"{storage_account}.dfs.core.windows.net"
adls_options = {
    f"fs.azure.account.auth.type.{fqdn}": "OAuth",
    f"fs.azure.account.oauth.provider.type.{fqdn}": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{fqdn}": client_id,
    f"fs.azure.account.oauth2.client.secret.{fqdn}": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{fqdn}": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}

GOLD = f"abfss://squad2@{fqdn}/fase2_ml/gold"
TABELAS = {nome: f"{GOLD}/item2vec_{nome}" for nome in
           ["skus", "cestas_treino", "holdout", "vetores", "recomendacoes_holdout", "avaliacao"]}

VERSAO = "v1_baseline"
TOP_K = 5
PARAMS = {                    # fixados no one page, antes de olhar o teste
    "vector_size": 32,        # vetor pequeno: só ~1,2 mil cestas de treino
    "window": 24,             # cobre a maior cesta: a ordem dos itens não tem significado
    "min_count": 1,           # o vocabulário (freq >= 2) já foi filtrado no notebook 02
    "sg": 1,                  # skip-gram: melhor para palavras (SKUs) raras
    "negative": 5,
    "epochs": 50,
    "seed": 42,
    "workers": 1,             # 1 thread: treino reprodutível
}

def ler_delta(caminho):
    return spark.read.format("delta").options(**adls_options).load(caminho)

def gravar_delta(df, destino, modo="overwrite"):
    w = df.write.format("delta").options(**adls_options).mode(modo)
    (w.option("overwriteSchema", "true") if modo == "overwrite" else w.option("mergeSchema", "true")).save(destino)

print(f"gensim {gensim.__version__} | versão do modelo: {VERSAO}")

## 1. Leitura das features
O treino do Word2Vec roda no driver (gensim é single-node e o Free Edition não tem MLlib). As três tabelas são pequenas (~1,2 mil frases, 336 cestas de teste e 3,8 mil SKUs), então vão para pandas.

As frases são **ordenadas por `id_pedido`**: o gensim inicializa os vetores na ordem em que vê os SKUs, e a ordem de leitura de uma tabela Delta não é garantida. Com a ordem fixa, a mesma `seed` dá sempre o mesmo modelo.

In [ ]:
skus = ler_delta(TABELAS["skus"]).toPandas()
treino = ler_delta(TABELAS["cestas_treino"]).orderBy("id_pedido").toPandas()
holdout = ler_delta(TABELAS["holdout"]).orderBy("id_pedido").toPandas()

frases = [list(s) for s in treino["skus"]]
print(f"Frases de treino: {len(frases):,} | tokens: {sum(map(len, frases)):,} | "
      f"cestas de teste: {len(holdout):,} | SKUs no catálogo: {len(skus):,}")

## 2. Popularidade e fallback
Duas listas saem do treino:
* **mais vendidos geral:** o baseline ingênuo e o último recurso de qualquer recomendação;
* **mais vendidos por subcategoria:** o fallback para quando o Item2Vec não tem vetor para os itens do carrinho, ou devolve menos de 5 sugestões.

In [ ]:
populares = skus[skus["freq_treino"] > 0].sort_values(["freq_treino", "sku"], ascending=[False, True])
mais_vendidos = populares["sku"].tolist()
mais_vendidos_sub = populares.groupby("subcategoria", sort=False)["sku"].apply(list).to_dict()
subcategoria_de = dict(zip(skus["sku"], skus["subcategoria"]))

def completar(recs, carrinho, candidatos):
    """Acrescenta candidatos que não estão no carrinho nem já recomendados, até TOP_K."""
    for sku in candidatos:
        if len(recs) >= TOP_K:
            break
        if sku not in carrinho and sku not in recs:
            recs.append(sku)
    return recs

def recomendar_populares(carrinho):
    return completar([], set(carrinho), mais_vendidos)

print("Top-5 mais vendidos no treino:", mais_vendidos[:TOP_K])

## 3. Treino do Item2Vec
Cada frase é uma cesta e cada palavra um SKU. O skip-gram aprende, para cada SKU, um vetor que prevê os outros SKUs da mesma cesta: produtos comprados juntos terminam com vetores parecidos.

In [ ]:
inicio = datetime.now()
modelo = Word2Vec(sentences=frases, **PARAMS)
print(f"Treino em {(datetime.now() - inicio).total_seconds():.1f}s | "
      f"vocabulário: {len(modelo.wv):,} SKUs × {modelo.wv.vector_size} dimensões")

## 4. Como o modelo recomenda
1. Pega os vetores dos itens do carrinho que estão no vocabulário e calcula a **média** (o "perfil" do carrinho).
2. Busca os SKUs mais próximos desse perfil por **similaridade de cosseno** e descarta o que já está no carrinho.
3. Se faltar sugestão (nenhum item do carrinho tem vetor, por exemplo), completa com os **mais vendidos da subcategoria** dos itens do carrinho e, por último, com os mais vendidos geral.

In [ ]:
def recomendar_item2vec(carrinho, usar_fallback=True):
    no_carrinho = set(carrinho)
    conhecidos = [s for s in carrinho if s in modelo.wv]
    recs = []
    if conhecidos:
        perfil = np.mean([modelo.wv[s] for s in conhecidos], axis=0)
        for sku, _ in modelo.wv.similar_by_vector(perfil, topn=TOP_K + len(conhecidos)):
            if sku not in no_carrinho:
                recs.append(sku)
            if len(recs) == TOP_K:
                break
    if usar_fallback:
        for sub in dict.fromkeys(subcategoria_de.get(s) for s in carrinho):
            recs = completar(recs, no_carrinho, mais_vendidos_sub.get(sub, []))
        recs = completar(recs, no_carrinho, mais_vendidos)
    return recs

exemplo = holdout.iloc[0]
print("Carrinho:", list(exemplo["carrinho"]))
print("Item removido (resposta):", exemplo["item_removido"])
print("Top-5 do Item2Vec:", recomendar_item2vec(list(exemplo["carrinho"])))

## 5. Avaliação no hold-out
* **Hit@5:** % das cestas em que o item removido aparece entre as 5 sugestões.
* **Precision@5:** acertos ÷ 5. Como só há 1 item certo por cesta, é sempre Hit@5 ÷ 5.
* **Cobertura do catálogo:** % dos SKUs do catálogo que aparecem em alguma recomendação. Um modelo que só recomenda os mesmos 5 produtos tem cobertura quase zero.
* **Acerto de subcategoria** (diagnóstico): % das cestas em que alguma sugestão é da mesma subcategoria do item removido. Mostra se o modelo "entende" a missão de compra mesmo quando erra o SKU exato.

In [ ]:
def avaliar(nome, recomendar):
    linhas, acertos, acertos_sub, recomendados = [], 0, 0, set()
    for _, cesta in holdout.iterrows():
        recs = recomendar(list(cesta["carrinho"]))
        acerto = cesta["item_removido"] in recs
        acertos += acerto
        acertos_sub += subcategoria_de.get(cesta["item_removido"]) in {subcategoria_de.get(r) for r in recs}
        recomendados.update(recs)
        linhas.append({"id_pedido": int(cesta["id_pedido"]), "modelo": nome, "recomendacoes": recs,
                       "item_removido": cesta["item_removido"], "acerto": bool(acerto)})
    n = len(holdout)
    metricas = {"modelo": nome, "cestas": n, "hit_at_5": acertos / n, "precision_at_5": acertos / n / TOP_K,
                "cobertura_catalogo": len(recomendados) / len(skus), "skus_recomendados": len(recomendados),
                "acerto_subcategoria": acertos_sub / n}
    return metricas, linhas

resultados = [avaliar("top5_mais_vendidos", recomendar_populares),
              avaliar("item2vec_sem_fallback", lambda c: recomendar_item2vec(c, usar_fallback=False)),
              avaliar("item2vec_com_fallback", recomendar_item2vec)]
metricas = pd.DataFrame([m for m, _ in resultados])

exibir = metricas.copy()
for c in ["hit_at_5", "precision_at_5", "cobertura_catalogo", "acerto_subcategoria"]:
    exibir[c] = (100 * exibir[c]).round(2).astype(str) + "%"
display(exibir)

base = metricas.set_index("modelo").loc["top5_mais_vendidos", "hit_at_5"]
i2v = metricas.set_index("modelo").loc["item2vec_com_fallback", "hit_at_5"]
print(f"Item2Vec com fallback acerta {i2v / base:.1f}x mais que o top-5 mais vendidos.")

## 6. O que o modelo aprendeu
Vizinhos mais próximos de alguns SKUs populares, com a subcategoria de cada um. É a verificação de bom senso: os vizinhos devem pertencer à mesma "missão de compra" vista na EDA (despensa, café da manhã, casa, bebê).

In [ ]:
amostra = [s for s in mais_vendidos if s in modelo.wv][:3]
vizinhos = [{"sku": s, "subcategoria": subcategoria_de.get(s), "vizinho": v,
             "subcategoria_vizinho": subcategoria_de.get(v), "cosseno": round(float(c), 3)}
            for s in amostra for v, c in modelo.wv.most_similar(s, topn=5)]
display(pd.DataFrame(vizinhos))

## 7. Gravação na Gold
* `item2vec_vetores`: o modelo em si, um vetor por SKU do vocabulário, com a versão e os parâmetros. É o que a inferência em streaming vai carregar.
* `item2vec_recomendacoes_holdout`: o top-5 de cada carrinho de teste, por modelo, para auditar acertos e erros.
* `item2vec_avaliacao`: uma linha por modelo e versão, em **append**, para acompanhar a evolução das melhorias.

In [ ]:
treinado_em = datetime.now()
vetores = pd.DataFrame({
    "sku": modelo.wv.index_to_key,
    "vetor": [modelo.wv[s].astype(float).tolist() for s in modelo.wv.index_to_key],
})
vetores["versao"] = VERSAO
vetores["params"] = str(PARAMS)
vetores["treinado_em"] = treinado_em
gravar_delta(spark.createDataFrame(vetores), TABELAS["vetores"])

recs_pdf = pd.DataFrame([l for _, linhas in resultados for l in linhas])
recs_pdf["versao"] = VERSAO
gravar_delta(spark.createDataFrame(recs_pdf), TABELAS["recomendacoes_holdout"])

aval = metricas.copy()
aval["versao"] = VERSAO
aval["params"] = str(PARAMS)
aval["avaliado_em"] = treinado_em
gravar_delta(spark.createDataFrame(aval), TABELAS["avaliacao"], modo="append")

print(f"Vetores: {ler_delta(TABELAS['vetores']).count():,} SKUs | "
      f"recomendações: {ler_delta(TABELAS['recomendacoes_holdout']).count():,} linhas | "
      f"avaliações acumuladas: {ler_delta(TABELAS['avaliacao']).count():,}")

## 8. Resultado do baseline

> Valores medidos localmente com os mesmos dados e parâmetros em 12/11/2026, antes da execução no Databricks. Conferir com a tabela da seção 5.

| Modelo | Hit@5 | Precision@5 | Cobertura | Acerto de subcategoria |
|---|---|---|---|---|
| Top-5 mais vendidos | 1,49% | 0,30% | 0,18% (7 SKUs) | 16,4% |
| Item2Vec sem fallback | 3,87% | 0,77% | 15,6% | 34,8% |
| **Item2Vec com fallback** | **4,17%** | **0,83%** | **17,8%** | **37,2%** |
| *FP-Growth (Lucas, outro hold-out)* | *1,2%* | — | *~11%* | — |

**Leitura:**
* O Item2Vec acerta **2,8× mais** que os mais vendidos e recomenda 17,8% do catálogo, contra 7 SKUs do baseline ingênuo.
* Em 37% das cestas ele sugere algo da **mesma subcategoria** do item que faltava: o modelo capturou as missões de compra, mesmo quando erra o SKU exato.
* O Hit@5 absoluto é baixo porque a base é pequena e esparsa: 30% dos SKUs aparecem uma única vez e 29% dos itens de teste nem têm vetor (teto de 70,8% do notebook 02).
* A comparação com o FP-Growth só vale depois que os dois rodarem no **mesmo** hold-out.

**Melhorias para as próximas versões** (sempre medidas contra esta linha):
1. Separar uma **validação dentro do treino** para ajustar `epochs`, `vector_size` e `window` sem tocar no teste. Em teste exploratório, mais épocas (200) chegaram a ~6% de Hit@5.
2. Testar média ponderada do carrinho (pesar itens raros) e o fallback por categoria raiz.
3. Avaliar a variação entre `seeds` e reportar a média, não uma execução só.

**Próximo passo:** `04_inferencia_streaming`. Ler os micro-lotes novos da Silver de itens com `Trigger.AvailableNow`, carregar `item2vec_vetores` e gravar o top-5 de cada pedido.